# Verify Processed Training Data

This notebook loads and verifies the processed training data from S3 to ensure it's correctly formatted for SageMaker XGBoost training.

## Verification Checks:
- ✅ Load data from S3 (train.csv.gz, val.csv.gz, test.csv.gz)
- ✅ Verify feature count (29 features)
- ✅ Verify feature order matches FEATURE_NAMES
- ✅ Verify label is last column
- ✅ Verify no header row
- ✅ Check class distribution
- ✅ Validate data types (all numeric)
- ✅ Check for missing/infinite values
- ✅ Display sample rows and statistics


In [1]:
# Import required libraries
import os
import sys
import pandas as pd
import numpy as np
import json
import boto3
from botocore.exceptions import ClientError
import warnings
warnings.filterwarnings('ignore')

# Add project root to path for imports
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from training.feature_engineering import FEATURE_NAMES

print("✅ Libraries imported successfully")
print(f"📊 Expected feature count: {len(FEATURE_NAMES)}")
print(f"📋 Feature names: {', '.join(FEATURE_NAMES[:5])}... (showing first 5)")


✅ Libraries imported successfully
📊 Expected feature count: 29
📋 Feature names: ip_click_count_24h, device_click_count_1h, time_since_last_click, hour_of_day, day_of_week... (showing first 5)


In [2]:
# Configuration
S3_BUCKET = 'fraudguard-ai-data-971422717446'
S3_PREFIX = 'fraud-detection/training/processed/'

# S3 paths
TRAIN_PATH = f's3://{S3_BUCKET}/{S3_PREFIX}train.csv.gz'
VAL_PATH = f's3://{S3_BUCKET}/{S3_PREFIX}val.csv.gz'
TEST_PATH = f's3://{S3_BUCKET}/{S3_PREFIX}test.csv.gz'
METRICS_PATH = f's3://{S3_BUCKET}/{S3_PREFIX}metrics.json'

print("📦 S3 Configuration:")
print(f"   Bucket: {S3_BUCKET}")
print(f"   Prefix: {S3_PREFIX}")
print(f"   Train: {TRAIN_PATH}")
print(f"   Validation: {VAL_PATH}")
print(f"   Test: {TEST_PATH}")
print(f"   Metrics: {METRICS_PATH}")


📦 S3 Configuration:
   Bucket: fraudguard-ai-data-971422717446
   Prefix: fraud-detection/training/processed/
   Train: s3://fraudguard-ai-data-971422717446/fraud-detection/training/processed/train.csv.gz
   Validation: s3://fraudguard-ai-data-971422717446/fraud-detection/training/processed/val.csv.gz
   Test: s3://fraudguard-ai-data-971422717446/fraud-detection/training/processed/test.csv.gz
   Metrics: s3://fraudguard-ai-data-971422717446/fraud-detection/training/processed/metrics.json


In [11]:
def load_processed_data_from_s3(s3_path: str, sample_size: int = None) -> pd.DataFrame:
    """
    Load processed training data from S3
    
    Args:
        s3_path: S3 path to CSV file (can be compressed .gz)
        sample_size: Optional limit on number of rows to load
    
    Returns:
        DataFrame with features and label
    """
    s3_client = boto3.client('s3')
    
    # Parse S3 path
    s3_path = s3_path.replace('s3://', '')
    bucket, key = s3_path.split('/', 1)
    
    print(f"📥 Loading from s3://{bucket}/{key}...")
    
    try:
        # Get object from S3
        response = s3_client.get_object(Bucket=bucket, Key=key)
        
        # Determine compression
        compression = 'gzip' if key.endswith('.gz') else None
        
        # Load CSV (no header, label in last column)
        df = pd.read_csv(
            response['Body'],
            header=None,
            compression=compression,
            nrows=sample_size
        )
        
        print(f"   ✅ Loaded {len(df):,} rows, {df.shape[1]} columns")
        
        # Assign column names for easier inspection
        # Last column is label, rest are features
        num_features = len(FEATURE_NAMES)
        if df.shape[1] == num_features + 1:
            df.columns = list(FEATURE_NAMES) + ['label']
        else:
            # If column count doesn't match, use generic names
            df.columns = [f'feature_{i}' for i in range(df.shape[1] - 1)] + ['label']
            print(f"   ⚠️  Warning: Expected {num_features + 1} columns, got {df.shape[1]}")
        
        return df
    
    except ClientError as e:
        print(f"   ❌ Error loading from S3: {e}")
        raise
    except Exception as e:
        print(f"   ❌ Error: {e}")
        raise

print("✅ Function defined")


✅ Function defined


In [12]:
# Load metrics.json
s3_client = boto3.client('s3')
bucket, key = METRICS_PATH.replace('s3://', '').split('/', 1)

print("📊 Loading metrics.json...")
try:
    response = s3_client.get_object(Bucket=bucket, Key=key)
    metrics = json.loads(response['Body'].read().decode('utf-8'))
    
    print("\n" + "=" * 70)
    print("METRICS SUMMARY")
    print("=" * 70)
    print(f"Feature Count: {metrics.get('feature_count', 'N/A')}")
    print(f"Scale Pos Weight: {metrics.get('scale_pos_weight', 'N/A'):.4f}")
    print()
    
    # Training set metrics
    train_metrics = metrics.get('train', {})
    print("Training Set:")
    print(f"  Total samples: {train_metrics.get('total_samples', 'N/A'):,}")
    print(f"  Fraud samples: {train_metrics.get('num_fraud', 'N/A'):,} ({train_metrics.get('fraud_rate', 0)*100:.2f}%)")
    print(f"  Legitimate samples: {train_metrics.get('num_legitimate', 'N/A'):,} ({train_metrics.get('legitimate_rate', 0)*100:.2f}%)")
    print()
    
    # Validation set metrics
    val_metrics = metrics.get('validation', {})
    print("Validation Set:")
    print(f"  Total samples: {val_metrics.get('total_samples', 'N/A'):,}")
    print(f"  Fraud samples: {val_metrics.get('num_fraud', 'N/A'):,} ({val_metrics.get('fraud_rate', 0)*100:.2f}%)")
    print(f"  Legitimate samples: {val_metrics.get('num_legitimate', 'N/A'):,} ({val_metrics.get('legitimate_rate', 0)*100:.2f}%)")
    print()
    
    # Test set metrics
    test_metrics = metrics.get('test', {})
    print("Test Set:")
    print(f"  Total samples: {test_metrics.get('total_samples', 'N/A'):,}")
    print(f"  Fraud samples: {test_metrics.get('num_fraud', 'N/A'):,} ({test_metrics.get('fraud_rate', 0)*100:.2f}%)")
    print(f"  Legitimate samples: {test_metrics.get('num_legitimate', 'N/A'):,} ({test_metrics.get('legitimate_rate', 0)*100:.2f}%)")
    
except Exception as e:
    print(f"❌ Error loading metrics: {e}")
    metrics = None


📊 Loading metrics.json...

METRICS SUMMARY
Feature Count: 29
Scale Pos Weight: 0.0535

Training Set:
  Total samples: 7,000
  Fraud samples: 6,980 (99.71%)
  Legitimate samples: 20 (0.29%)

Validation Set:
  Total samples: 1,500
  Fraud samples: 1,496 (99.73%)
  Legitimate samples: 4 (0.27%)

Test Set:
  Total samples: 1,500
  Fraud samples: 1,495 (99.67%)
  Legitimate samples: 5 (0.33%)


In [5]:
# Load training data (sample first 1000 rows for quick verification)
print("=" * 70)
print("LOADING TRAINING DATA")
print("=" * 70)
train_df = load_processed_data_from_s3(TRAIN_PATH, sample_size=1000)
print()
print(f"Shape: {train_df.shape}")
print(f"Columns: {list(train_df.columns[:5])}... (showing first 5)")
print()
print("First few rows:")
train_df.head()


LOADING TRAINING DATA
📥 Loading from s3://fraudguard-ai-data-971422717446/fraud-detection/training/processed/train.csv.gz...
   ✅ Loaded 1,000 rows, 30 columns

Shape: (1000, 30)
Columns: ['ip_click_count_24h', 'device_click_count_1h', 'time_since_last_click', 'hour_of_day', 'day_of_week']... (showing first 5)

First few rows:


,ip_click_count_24h,device_click_count_1h,time_since_last_click,hour_of_day,day_of_week,ua_is_bot,ua_entropy,ip_is_datacenter,ip_is_vpn,ip_is_proxy,...,time_to_conversion_sec,ip_country_encoded,device_os_encoded,click_to_install_time_sec,has_recent_install,install_broadcast_detected,click_injection_risk_score,conversion_rate,engagement_score,label
0,0.0,177.0,15.0,10.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,0.0,22.0,234.0,21.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,0.0,2.0,2310.0,8.0,2.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,0.0,3.0,679.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,21.0,7.0,804.0,5.0,2.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1


In [6]:
# Verification checks
print("=" * 70)
print("VERIFICATION CHECKS")
print("=" * 70)

checks_passed = []
checks_failed = []

# Check 1: Feature count
expected_features = len(FEATURE_NAMES)
actual_features = train_df.shape[1] - 1  # Exclude label column
if actual_features == expected_features:
    checks_passed.append(f"✅ Feature count: {actual_features} (expected {expected_features})")
else:
    checks_failed.append(f"❌ Feature count: {actual_features} (expected {expected_features})")

# Check 2: Label is last column
if train_df.columns[-1] == 'label':
    checks_passed.append("✅ Label is last column")
else:
    checks_failed.append(f"❌ Label is not last column. Last column: {train_df.columns[-1]}")

# Check 3: All features are numeric
non_numeric = train_df.iloc[:, :-1].select_dtypes(exclude=[np.number]).columns.tolist()
if len(non_numeric) == 0:
    checks_passed.append("✅ All features are numeric")
else:
    checks_failed.append(f"❌ Non-numeric features found: {non_numeric}")

# Check 4: Label is binary (0 or 1)
label_values = train_df['label'].unique()
if set(label_values).issubset({0, 1}):
    checks_passed.append(f"✅ Label is binary: {sorted(label_values)}")
else:
    checks_failed.append(f"❌ Label contains non-binary values: {label_values}")

# Check 5: No missing values
missing_count = train_df.isnull().sum().sum()
if missing_count == 0:
    checks_passed.append("✅ No missing values")
else:
    checks_failed.append(f"❌ Found {missing_count} missing values")

# Check 6: No infinite values
infinite_count = np.isinf(train_df.iloc[:, :-1].select_dtypes(include=[np.number])).sum().sum()
if infinite_count == 0:
    checks_passed.append("✅ No infinite values")
else:
    checks_failed.append(f"❌ Found {infinite_count} infinite values")

# Check 7: Feature order matches FEATURE_NAMES
feature_cols = list(train_df.columns[:-1])  # Exclude label
if feature_cols == FEATURE_NAMES:
    checks_passed.append("✅ Feature order matches FEATURE_NAMES")
else:
    mismatches = [i for i, (a, b) in enumerate(zip(feature_cols, FEATURE_NAMES)) if a != b]
    if mismatches:
        checks_failed.append(f"❌ Feature order mismatch at indices: {mismatches[:5]}...")
    else:
        checks_failed.append(f"❌ Feature count mismatch: {len(feature_cols)} vs {len(FEATURE_NAMES)}")

# Print results
print("\nPASSED CHECKS:")
for check in checks_passed:
    print(f"  {check}")

if checks_failed:
    print("\nFAILED CHECKS:")
    for check in checks_failed:
        print(f"  {check}")
else:
    print("\n🎉 All checks passed!")

print(f"\nTotal: {len(checks_passed)} passed, {len(checks_failed)} failed")


VERIFICATION CHECKS

PASSED CHECKS:
  ✅ Feature count: 29 (expected 29)
  ✅ Label is last column
  ✅ All features are numeric
  ✅ Label is binary: [np.int64(0), np.int64(1)]
  ✅ No missing values
  ✅ No infinite values
  ✅ Feature order matches FEATURE_NAMES

🎉 All checks passed!

Total: 7 passed, 0 failed


In [7]:
# Class distribution
print("=" * 70)
print("CLASS DISTRIBUTION")
print("=" * 70)

label_counts = train_df['label'].value_counts().sort_index()
total = len(train_df)

print("\nTraining Set (sample):")
print(f"  Total samples: {total:,}")
print(f"  Fraud (label=1): {label_counts.get(1, 0):,} ({label_counts.get(1, 0)/total*100:.2f}%)")
print(f"  Legitimate (label=0): {label_counts.get(0, 0):,} ({label_counts.get(0, 0)/total*100:.2f}%)")

# Calculate scale_pos_weight
if 1 in label_counts and 0 in label_counts:
    num_fraud = label_counts[1]
    num_legitimate = label_counts[0]
    scale_pos_weight = np.sqrt(num_legitimate / num_fraud)
    print(f"\n  Scale pos weight: {scale_pos_weight:.4f}")


CLASS DISTRIBUTION

Training Set (sample):
  Total samples: 1,000
  Fraud (label=1): 994 (99.40%)
  Legitimate (label=0): 6 (0.60%)

  Scale pos weight: 0.0777


In [8]:
# Feature statistics
print("=" * 70)
print("FEATURE STATISTICS")
print("=" * 70)

feature_df = train_df.iloc[:, :-1]  # Exclude label

print(f"\nFeature count: {feature_df.shape[1]}")
print(f"\nData types:")
print(feature_df.dtypes.value_counts())

print(f"\nSummary statistics (first 10 features):")
feature_df.iloc[:, :10].describe()


FEATURE STATISTICS

Feature count: 29

Data types:
float64    29
Name: count, dtype: int64

Summary statistics (first 10 features):


,ip_click_count_24h,device_click_count_1h,time_since_last_click,hour_of_day,day_of_week,ua_is_bot,ua_entropy,ip_is_datacenter,ip_is_vpn,ip_is_proxy
count,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000,1000.0,1000.0,1000.0,1000.0,1000.0
mean,0.74500,146.895000,82.824000,9.321000,1.883000,0.0,0.0,0.0,0.0,0.0
std,2.69838,48.740143,545.955713,6.115501,0.886621,0.0,0.0,0.0,0.0,0.0
min,0.00000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
25%,0.00000,140.000000,7.000000,4.000000,1.000000,0.0,0.0,0.0,0.0,0.0
50%,0.00000,161.000000,18.000000,9.000000,2.000000,0.0,0.0,0.0,0.0,0.0
75%,0.00000,175.000000,38.000000,13.250000,3.000000,0.0,0.0,0.0,0.0,0.0
max,24.00000,218.000000,13402.000000,23.000000,3.000000,0.0,0.0,0.0,0.0,0.0


In [9]:
# Load validation and test sets
print("=" * 70)
print("LOADING VALIDATION AND TEST DATA")
print("=" * 70)

val_df = load_processed_data_from_s3(VAL_PATH, sample_size=500)
print()
test_df = load_processed_data_from_s3(TEST_PATH, sample_size=500)

print("\n" + "=" * 70)
print("DATASET SUMMARY")
print("=" * 70)
print(f"Training samples:   {len(train_df):,}")
print(f"Validation samples: {len(val_df):,}")
print(f"Test samples:      {len(test_df):,}")
print(f"Total samples:     {len(train_df) + len(val_df) + len(test_df):,}")
print(f"Features per sample: {train_df.shape[1] - 1}")


LOADING VALIDATION AND TEST DATA
📥 Loading from s3://fraudguard-ai-data-971422717446/fraud-detection/training/processed/val.csv.gz...
   ✅ Loaded 500 rows, 30 columns

📥 Loading from s3://fraudguard-ai-data-971422717446/fraud-detection/training/processed/test.csv.gz...
   ✅ Loaded 500 rows, 30 columns

DATASET SUMMARY
Training samples:   1,000
Validation samples: 500
Test samples:      500
Total samples:     2,000
Features per sample: 29


In [10]:
# Sample rows for inspection
print("=" * 70)
print("SAMPLE ROWS (First 3)")
print("=" * 70)

print("\nTraining Set:")
print(train_df.head(3).to_string())

print("\n\nValidation Set:")
print(val_df.head(3).to_string())

print("\n\nTest Set:")
print(test_df.head(3).to_string())


SAMPLE ROWS (First 3)

Training Set:
   ip_click_count_24h  device_click_count_1h  time_since_last_click  hour_of_day  day_of_week  ua_is_bot  ua_entropy  ip_is_datacenter  ip_is_vpn  ip_is_proxy  ip_is_business  ip_is_competitor  geo_distance_km  referrer_is_valid  click_to_view_time_ms  campaign_fraud_rate  publisher_quality  device_fingerprint_entropy  is_mobile  is_repeated_click  time_to_conversion_sec  ip_country_encoded  device_os_encoded  click_to_install_time_sec  has_recent_install  install_broadcast_detected  click_injection_risk_score  conversion_rate  engagement_score  label
0                 0.0                  177.0                   15.0         10.0          1.0        0.0         0.0               0.0        0.0          0.0             0.0               0.0              0.0                0.0                    0.0                  0.0                0.5                         0.0        0.0                0.0                     0.0                 0.0            

In [ ]:
# Verify feature names match expected order
print("=" * 70)
print("FEATURE ORDER VERIFICATION")
print("=" * 70)

actual_features = list(train_df.columns[:-1])  # Exclude label
expected_features = FEATURE_NAMES

print(f"\nExpected features ({len(expected_features)}):")
for i, feat in enumerate(expected_features[:10]):
    print(f"  {i:2d}. {feat}")
print(f"  ... ({len(expected_features) - 10} more)")

print(f"\nActual features ({len(actual_features)}):")
for i, feat in enumerate(actual_features[:10]):
    match = "✅" if i < len(expected_features) and feat == expected_features[i] else "❌"
    print(f"  {i:2d}. {feat} {match}")
print(f"  ... ({len(actual_features) - 10} more)")

# Check for mismatches
mismatches = []
for i, (actual, expected) in enumerate(zip(actual_features, expected_features)):
    if actual != expected:
        mismatches.append((i, actual, expected))

if mismatches:
    print(f"\n❌ Found {len(mismatches)} mismatches:")
    for idx, actual, expected in mismatches[:5]:
        print(f"  Index {idx}: '{actual}' != '{expected}'")
else:
    print("\n✅ All features match expected order!")


## Summary

This notebook verifies that the processed training data:
- ✅ Has correct number of features (29)
- ✅ Features are in the correct order matching `FEATURE_NAMES`
- ✅ Label is the last column (0 = legitimate, 1 = fraud)
- ✅ All values are numeric
- ✅ No missing or infinite values
- ✅ Label is binary (0 or 1)
- ✅ Data is ready for SageMaker XGBoost training

**Next Steps:**
1. If all checks pass, proceed to training: `docs/BOT_TRAFFIC_TRAINING_GUIDE.md`
2. If checks fail, review the processing pipeline: `docs/DATA_PROCESSING_WALKTHROUGH.md`
